### 프롬프트 템플릿  Agent tool rag에서 표준
- role:LLM이 어떤 역활을 할지 정함
- instruction:답변 규칙과 형식을 정리
- context:데이터베이스 검색결과처럼 답변에 참고할 실제 정보
- query:실제 질문


In [19]:
# 사용자 질문 -> LLM개입해서 사용자 의도를 파악해서
# 적절한 tool 호출 -> 라우터( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [20]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')

class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,
            response_format={'type':'json_object'}
        )
        return response.choices[0].message.content
llm = OpenAILLM()
llm.model  

'gpt-5.4-nano'

In [21]:
# 데이터
do_search_results = [
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""
for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [22]:
from  textwrap import dedent  #  들여쓰기 indent를 제거
import json
# 프롬프트 템플릿 - 고정된 구조
# 프롬프트생성 + llm호출 + 파싱
def recommend_product(user_question:str, context:str) ->dict:    
    prompt = dedent(f'''
                        당신은 이커머스 전문 AI 추천 어시스턴트입니다.
                        아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
                        반드시 json 형식으로만 답하세요

                        [컨텍스트]
                        {context}

                        [질문]
                        {user_question}

                        답변은 반드시 아래 형식의 json으로만 출력하세요
                        {
                            {
                                "recommended_product" : "상품명",
                                "reason":"추천사유"                            
                            }
                        }
                        ''')
    response = llm.generate(prompt)
    data = json.loads(response)
    return json.dumps(data, indent=2, ensure_ascii=False)
    
    

### 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [23]:
def route_qeustion(question:str)->str:
    lower_question = question.lower()
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):
        return "llm_recomendation"
    else:
        return "general_llm"
sample_questions = [
    "3개 상품을 2개씩 주문하면 총 몇 개인가?",
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘.",
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘.",
    "코딩할 때 쓸만한 노트북 추천해줘.",
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_qeustion(question)}')    

질문 : 3개 상품을 2개씩 주문하면 총 몇 개인가? | 라우트 : calculator_tool
질문 : 나는 코딩용 노트북을 좋아한다는 점을 기억해줘. | 라우트 : memory_tool
질문 : AI 에이전트 뉴스 최신 기사 3개 찾아줘. | 라우트 : news_tool
질문 : 코딩할 때 쓸만한 노트북 추천해줘. | 라우트 : llm_recomendation


### tool 활용

In [24]:
import json
import os
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calcualtor_tool(text:str)->float:
    allowed_chars = set("0123456789+-*/().")
    if not set(text) <= allowed_chars: # text의 문자중에 허용되지 않은 문자가 있다면
        raise ValueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)

In [25]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)

        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [26]:
print(calcualtor_tool('30*50'))
search_naver_news('AI 에이전트')

1500


[{'title': 'AI고객센터 운영하는 유베이스, 음성인식 스타트업 리턴제로 인수',
  'content': '또 AI 기술을 이용해 고객 상담의 전 과정을 스스로 처리하는 완성형 상담 AI 에이전트도 7월에 공개할 예정이다. 목진원 유베이스 대표는 "리턴제로의 합류로 음성 AI 기술까지 갖춘 만큼 다른 업체들과 기술 격차를 더... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/469/0000932886?sid=105'},
 {'title': "카카오, 학계와 AI 연구 협력 확대…'카나나 스칼라' 콜로키움 개최",
  'content': '주요 발표 주제는 ▲LLM 에이전트 추론 효율 및 신뢰성 향상 ▲멀티모달 AI 안전성 강화 ▲연합학습 기반 모델 개인화 ▲초장문 영상 이해 ▲실시간 립싱크 생성 ▲3D 비전 및 인간-물체 상호작용 모델링 등이다. 카카오... ',
  'date': '2026-05-27',
  'link': 'https://n.news.naver.com/mnews/article/003/0013969771?sid=105'},
 {'title': "LG이노텍, '2026 ECTC'서 AI용 대면적 FC-BGA 기판 공개",
  'content': '학습·추론형 AI 확산, AI에이전트의 토큰 사용량 급증에 따라 반도체 칩의 성능이 고도화되는 추세에서 고사양 반도체 칩이 대량의 데이터를 빠른 속도로 처리하기 위해서는 더 많은 회로와 부품을 기판에... ',
  'date': '2026-05-27',
  'link': 'https://www.itbiznews.com/news/articleView.html?idxno=214121'}]

### 메모리 활용하기
- 이전대화나 사용자의 선호를 저장해서 다음 응답에 반영하는 기능

In [27]:
session_memory={}
def remember_preference(user_id:str, key:str, value:str)->None:
    if user_id not in session_memory:
        session_memory[user_id] = {}
    session_memory[user_id][key] = value

def get_preference(user_id:str, key:str, default:str | None = None) -> str | None:
    return session_memory.get(user_id,{}).get(key,default)

user_id = 'student-001'
remember_preference(user_id, 'category','노트북')
remember_preference(user_id, 'usage','코딩')

print(session_memory)

{'student-001': {'category': '노트북', 'usage': '코딩'}}


### 라우터 + 도구 + 메모리 통합
- 라우팅 규칙, 계산도구, 네이버뉴스도구, 메모리 저장을 하나의 흐름으로 연결--> Agent의 기본 형태
- 먼저 질문을 분류하고 그 분류 결과에 맞는 도구를 호출한뒤 필요하면 메모리까지 갱신
- 최종 결과를 llm에 전달해서 답변을 생성

In [28]:
def extract_math_expression(question: str) -> str:
    match = re.search(r"[0-9\s\+\-\*\/\(\)\.]+", question)
    if not match:
        raise ValueError("계산식을 찾을 수 없습니다.")
    expression = match.group(0).strip()
    if not expression:
        raise ValueError("계산식이 비어 있습니다.")
    return expression

def extract_news_query(question: str) -> str:
    cleaned = re.sub(r"뉴스|기사|검색|알려줘|찾아줘|추천해줘|좀|최근|최신|오늘", " ", question)
    cleaned = re.sub(r"\d+\s*개?", " ", cleaned)
    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or question

def route_and_execute(question: str) -> dict:
    route = route_qeustion(question)

    if route == "news_tool":
        news_query = extract_news_query(question)
        news_result = search_naver_news(news_query, display=3)
        return {
            "route": route,
            "tool": "search_naver_news",
            "input": news_query,
            "result": news_result,
        }

    if route == "calculator_tool":
        expression = extract_math_expression(question)
        result = calcualtor_tool(expression)
        return {
            "route": route,
            "tool": "calculator_tool",
            "input": expression,
            "result": result,
        }

    if route == "memory_tool":
        remember_preference("student-001", "last_question", question)
        return {
            "route": route,
            "tool": "memory_tool",
            "input": question,
            "result": get_preference("student-001", "last_question"),
        }

    if route == "llm_recommendation":
        recommendation = recommend_product(question, context_string)
        return {
            "route": route,
            "tool": "llm_recommendation",
            "input": question,
            "result": recommendation,
        }

    return {
        "route": route,
        "tool": "general_llm",
        "input": question,
        "result": question,
    }

In [30]:
demo_questions = [
    '3 + 2 * 4',
    '어제 오늘 사고뉴스 5개 찾아줘',
    '점심메뉴 추천해줘'
]
for question in demo_questions:
    resutl = route_and_execute(question)
    print(f'질문:{question}')
    print(f'라우트:{resutl["route"]}')
    print(f'도구:{resutl["tool"]}')

질문:3 + 2 * 4
라우트:general_llm
도구:general_llm
질문:어제 오늘 사고뉴스 5개 찾아줘
라우트:news_tool
도구:search_naver_news
질문:점심메뉴 추천해줘
라우트:llm_recomendation
도구:general_llm
